In [1]:
# LazyStudent AI - turns messy notes into a revision pack
#
# --- Colab setup (run in the first cell) ---
!pip install -q transformers accelerate "gradio>=5,<6" pypdf huggingface_hub

from huggingface_hub import login
login()   # This will prompt you to enter your Hugging Face Access Token

# Runtime > Change runtime type > T4 GPU
# Then paste this file into the next cell and run.

import html
import re

import torch
import gradio as gr
from pypdf import PdfReader
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "google/gemma-2-2b-it"   # swap for a bigger Gemma if your GPU allows
MAX_CHARS = 6000                    # keeps prompts small for the 2B model

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)

PROMPTS = {
    "summary": (
        "Summarize the study notes below in simple English in 5-7 short bullet points. "
        "Only use information from the notes."
    ),
    "viva": (
        "Write 6 viva questions with short, correct answers based ONLY on the notes below. "
        "Format exactly as:\nQ1: ...\nA1: ...\nQ2: ...\nA2: ..."
    ),
    "flashcards": (
        "Create 8 flashcards from the notes below. One per line, format exactly:\n"
        "Term or question :: short answer"
    ),
    "checklist": (
        "Make a revision checklist of 8-10 topics a student must revise from the notes below. "
        "One per line, each starting with '- [ ] '."
    ),
}


def generate(instruction: str, notes: str, max_new_tokens: int = 500) -> str:
    messages = [{"role": "user", "content": f"{instruction}\n\nNOTES:\n{notes}"}]
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def read_file(path: str) -> str:
    if path.lower().endswith(".pdf"):
        return "\n".join((p.extract_text() or "") for p in PdfReader(path).pages)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


# ---------- parsing model output into UI pieces ----------
def clean(s: str) -> str:
    return re.sub(r"[*_`#]+", "", s).strip()


def parse_cards(text: str):
    cards = []
    for line in text.splitlines():
        if "::" in line:
            a, b = line.split("::", 1)
            a, b = clean(re.sub(r"^\s*[-\d.)]+\s*", "", a)), clean(b)
            if a and b:
                cards.append((a, b))
    return cards


def parse_checklist(text: str):
    items = []
    for line in text.splitlines():
        t = clean(re.sub(r"^\s*(-\s*\[.?\]|[-*\d.)])\s*", "", line))
        if t:
            items.append(t)
    return items


def cards_html(cards) -> str:
    if not cards:
        return "<p>No flashcards could be read from the model output. Try again.</p>"
    out = ['<p class="hint">Click a card to flip it.</p><div class="ls-cards">']
    for i, (q, a) in enumerate(cards):
        out.append(
            f'<label class="ls-card"><input type="checkbox" aria-label="Flip card {i+1}">'
            f'<span class="in"><span class="f">{html.escape(q)}</span>'
            f'<span class="b">{html.escape(a)}</span></span></label>'
        )
    out.append("</div>")
    return "".join(out)


def progress_html(done: int, total: int) -> str:
    pct = int(done / total * 100) if total else 0
    return (
        f'<div class="ls-prog">{done} of {total} revised</div>'
        f'<div class="ls-bar"><i style="width:{pct}%"></i></div>'
    )


def build_pack(text, file):
    parts = [text or ""]
    if file is not None:
        parts.append(read_file(file if isinstance(file, str) else file.name))
    notes = "\n".join(p for p in parts if p.strip()).strip()
    if not notes:
        raise gr.Error("Paste some notes or upload a PDF/TXT file first.")
    notes = notes[:MAX_CHARS]

    summary = generate(PROMPTS["summary"], notes, 350)
    viva = generate(PROMPTS["viva"], notes, 700)
    cards_raw = generate(PROMPTS["flashcards"], notes, 500)
    check_raw = generate(PROMPTS["checklist"], notes, 300)

    cards = parse_cards(cards_raw)
    items = parse_checklist(check_raw)

    md = (
        "# LazyStudent Study Pack\n\n"
        f"## Summary\n{summary}\n\n"
        f"## Viva Questions\n{viva}\n\n"
        "## Flashcards\n" + "\n".join(f"{q} :: {a}" for q, a in cards) + "\n\n"
        "## Revision Checklist\n" + "\n".join(f"- [ ] {t}" for t in items) + "\n"
    )
    path = "study_pack.md"
    with open(path, "w", encoding="utf-8") as f:
        f.write(md)

    return (
        summary,
        viva,
        cards_html(cards),
        gr.CheckboxGroup(choices=items, value=[], label="Tick what you have revised"),
        progress_html(0, len(items)),
        len(items),
        path,
    )


def update_progress(checked, total):
    return progress_html(len(checked or []), total or 0)


# ---------- look and feel ----------
CSS = """
.gradio-container{background-image:linear-gradient(rgba(120,150,190,.18) 1px,transparent 1px),
linear-gradient(90deg,rgba(120,150,190,.18) 1px,transparent 1px);background-size:28px 28px}
#title h1{font-size:2.4rem;font-weight:800;letter-spacing:-.02em;margin:0}
#title mark{background:#ffe45c;color:#14213d;padding:0 .15em;border-radius:3px}
.ls-cards{display:grid;grid-template-columns:repeat(auto-fill,minmax(190px,1fr));gap:12px}
.ls-card{display:block;height:130px;perspective:700px;cursor:pointer;position:relative}
.ls-card input{position:absolute;opacity:0;inset:0;cursor:pointer;z-index:2}
.ls-card .in{position:relative;display:block;width:100%;height:100%;transition:transform .45s;transform-style:preserve-3d}
.ls-card input:checked ~ .in{transform:rotateY(180deg)}
.ls-card input:focus-visible ~ .in{outline:3px solid #ffe45c;outline-offset:2px;border-radius:10px}
.ls-card .f,.ls-card .b{position:absolute;inset:0;backface-visibility:hidden;display:flex;align-items:center;
justify-content:center;text-align:center;padding:12px;font-size:.9rem;border:1.5px solid #14213d;border-radius:10px;
background:var(--background-fill-primary)}
.ls-card .f{font-weight:700}
.ls-card .b{transform:rotateY(180deg);background:#ffe45c;color:#14213d;font-weight:600}
.ls-bar{height:6px;background:rgba(120,150,190,.35);border-radius:9px;overflow:hidden;margin:4px 0 10px}
.ls-bar i{display:block;height:100%;background:#1f9d62;transition:width .25s}
.ls-prog{font-size:.85rem;opacity:.75}
.hint{font-size:.85rem;opacity:.7}
@media (prefers-reduced-motion:reduce){.ls-card .in,.ls-bar i{transition:none}}
"""

theme = gr.themes.Base(
    primary_hue=gr.themes.colors.yellow,
    neutral_hue=gr.themes.colors.slate,
    font=[gr.themes.GoogleFont("Bricolage Grotesque"), "system-ui", "sans-serif"],
    radius_size=gr.themes.sizes.radius_md,
).set(
    button_primary_background_fill="#ffe45c",
    button_primary_background_fill_hover="#ffd92e",
    button_primary_text_color="#14213d",
    button_primary_border_color="#14213d",
    block_border_width="1.5px",
)

with gr.Blocks(title="LazyStudent AI", theme=theme, css=CSS) as demo:
    gr.HTML('<div id="title"><h1>Lazy<mark>Student</mark> AI</h1></div>')
    gr.Markdown("Paste messy notes or upload a PDF/TXT. Get a summary, viva questions, flashcards and a revision checklist.")
    with gr.Row():
        with gr.Column():
            text = gr.Textbox(lines=14, label="Paste notes")
            file = gr.File(label="...or upload PDF / TXT", file_types=[".pdf", ".txt"])
            btn = gr.Button("Make my study pack", variant="primary")
        with gr.Column():
            with gr.Tab("Summary"):
                o_sum = gr.Markdown()
            with gr.Tab("Viva Q&A"):
                o_viva = gr.Markdown()
            with gr.Tab("Flashcards"):
                o_cards = gr.HTML()
            with gr.Tab("Checklist"):
                o_prog = gr.HTML()
                o_check = gr.CheckboxGroup(choices=[], label="Tick what you have revised")
            dl = gr.File(label="Download study pack (.md)")
    total = gr.State(0)

    btn.click(build_pack, [text, file], [o_sum, o_viva, o_cards, o_check, o_prog, total, dl])
    o_check.change(update_progress, [o_check, total], [o_prog])

demo.launch(share=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.6/325.6 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.4/462.4 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 6.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.14.0 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.52.1 which

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

/tmp/ipykernel_1452/1072492722.py:192: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="LazyStudent AI", theme=theme, css=CSS) as demo:
/tmp/ipykernel_1452/1072492722.py:192: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="LazyStudent AI", theme=theme, css=CSS) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://454a4da65e81bf1e90.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
